In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from tensorflow.keras.optimizers import Adam

# Hyperparameters
latent_dim = 10  # Dimension of the latent space
data_points = 500  # Number of real data points
epochs = 2000
batch_size = 32

# Generate real data (a Gaussian distribution)
real_data = np.random.normal(loc=0, scale=1, size=(data_points, 1))  # Real data

# Build the Generative Model
generator = Sequential([
    Dense(16, input_dim=latent_dim),
    LeakyReLU(0.2),
    Dense(32),
    LeakyReLU(0.2),
    Dense(1, activation='linear')  # Output: synthetic data
])

# Build the Discriminative Model
discriminator = Sequential([
    Dense(32, input_dim=1),
    LeakyReLU(0.2),
    Dense(16),
    LeakyReLU(0.2),
    Dense(1, activation='sigmoid')  # Output: probability (real or fake)
])

# Compile the discriminator
discriminator.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy', metrics=['accuracy'])

# Build the combined GAN model
discriminator.trainable = False
gan = Sequential([generator, discriminator])
gan.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy')

# Training the GAN
losses = []
for epoch in range(epochs):
    # Train the discriminator
    # Generate fake data
    noise = np.random.normal(0, 1, size=(batch_size, latent_dim))
    fake_data = generator.predict(noise, verbose=0)
    real_labels = np.ones((batch_size, 1))  # Real labels
    fake_labels = np.zeros((batch_size, 1))  # Fake labels

    # Train on real data
    d_loss_real = discriminator.train_on_batch(real_data[:batch_size], real_labels)
    # Train on fake data
    d_loss_fake = discriminator.train_on_batch(fake_data, fake_labels)

    # Train the generator (via the combined model)
    g_loss = gan.train_on_batch(noise, real_labels)

    # Save losses
    losses.append((d_loss_real[0], d_loss_fake[0], g_loss))

    # Print progress
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch + 1}, D Loss Real: {d_loss_real[0]:.4f}, D Loss Fake: {d_loss_fake[0]:.4f}, G Loss: {g_loss:.4f}")

# Visualizing the Results
# Generate new synthetic data
noise = np.random.normal(0, 1, size=(data_points, latent_dim))
generated_data = generator.predict(noise, verbose=0)

# Plot real vs generated data
plt.figure(figsize=(12, 6))
plt.hist(real_data, bins=30, alpha=0.7, label='Real Data', color='blue')
plt.hist(generated_data, bins=30, alpha=0.7, label='Generated Data', color='orange')
plt.title('Generative Model: Real vs Generated Data', fontsize=16)
plt.xlabel('Data Values', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.show()
